# Dynamic AI Product Evolution — Master Pipeline Notebook

**Canonical workflow control room**  
Repository scope: dated official sources → product–capability–task universe → longitudinal transitions → dated task measurement → aggregation and analysis.

This notebook is deliberately both:

1. a detailed, readable map of the thesis workflow; and
2. a safe orchestration interface that calls versioned scripts and reusable package code.

It is **not** the place where extraction rules or scoring logic are invented ad hoc. Those rules belong in the relevant spec, prompt, schema, implementation module, test, and eval case.

## How to use this notebook

Run the notebook from top to bottom whenever you begin a new project session.

In its default `status` mode it will not call external services or execute unfinished pipeline stages. It will:

- locate the repository root;
- load the project and stage registries;
- validate paths and stage definitions;
- show every stage, its governing file, expected inputs and outputs, and current readiness;
- prepare the commands that will eventually run the pipeline;
- explain the acceptance gates between phases.

Once a stage is implemented and passes its sentinel evals, update its status in `configs/pipeline_stages.yaml`. Then select it in the parameter cell below and switch to `selected` execution mode.

## Architectural rule: the notebook orchestrates; scripts implement

```text
MASTER NOTEBOOK
  explains dependencies, selects stages, performs preflight,
  invokes scripts, records results, displays gates
        ↓
NUMBERED PIPELINE SCRIPTS
  thin command-line entry points
        ↓
REUSABLE PACKAGE MODULES IN src/
  source discovery, normalization, extraction, matching,
  measurement, evaluation, aggregation, analysis
        ↓
IMMUTABLE RUN OUTPUTS + EVAL REPORTS
```

A notebook-only implementation would create hidden state, copy-pasted logic, and difficult-to-test behavior. For that reason, production logic must be importable and testable outside the notebook.

## End-to-end research workflow

```text
00 Company universe
  → 01 SEC source discovery
  → 02 Official-web source discovery
  → 03 Immutable snapshots
  → 04 Normalized documents and passages

  → 05 Product extraction
  → 06 Capability extraction
  → 07 Customer-facing task extraction
  → 08 Task-role classification

  → 09 Longitudinal entity resolution
  → 10 Task-transition classification

  → 11 Dated frontier capability registry
  → 12 Task-year measurement
       • frontier task replicability
       • AI transformation depth
       • deployment scale
       • task-specific defensibility

  → 13 Product- and firm-year aggregation
  → 14 Descriptive, comparative, and later outcome analysis
```

The eval harness is not a final appendix. It provides gates throughout the workflow.

## Current execution plan — thesis-first path

The canonical current record is the versioned planning pack:
[`docs/planning/README.md`](../docs/planning/README.md) and
[`docs/planning/01_CURRENT_STATUS_AND_EXECUTION_ROADMAP.md`](../docs/planning/01_CURRENT_STATUS_AND_EXECUTION_ROADMAP.md).
Read those before running any stage below; they state what has been established,
what the current gate is, and the required order from here.
[`docs/THESIS_EXECUTION_PLAN.md`](../docs/THESIS_EXECUTION_PLAN.md) remains useful
for the longer-range design argument, but parts of it predate the current state
and it is no longer the record of where the work stands.

```text
FRAME_v1  ->  UNIVERSE_v1  ->  EXTERNAL_SET_COMPARISON_v1  ->  SAMPLE_v1  ->  PCT_v1
```

Three points govern how the numbers in this notebook should be read:

- **Counts, costs and the final sample size are measured at gates**, not carried
  as fixed estimates. Any figure not yet produced by a manifested artefact is
  unverified, and a gate that would consume an unmeasured number stops instead.
- **Full-frame screening and classification are not full-frame extraction.**
  Screening runs over the whole frame because it is cheap; product-capability-task
  extraction runs only over `SAMPLE_v1`.
- **This notebook orchestrates; it does not implement.** No business logic, prompt
  text, taxonomy rule or tier derivation belongs in these cells
  (`specs/SPEC-026-master-notebook-orchestration.md`, required behaviour 6).

The publication-facing notebook is `01_reproduce_published_results.ipynb`, which
reproduces released tables and figures offline from `thesis_release_manifest.json`.
It does not exist yet and is created at the release-lock gate.

## 0. Environment setup

From a terminal in the repository root:

```bash
python -m pip install -e '.[dev,notebook]'
```

Then open this file in VS Code or JupyterLab. Select the environment in which the package was installed.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# Make the src-layout package importable even before editable installation.
def _find_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "CLAUDE.md").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the repository.")

REPO_ROOT = _find_root(Path.cwd())
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dynamic_ai_products.workflow import (
    create_notebook_run_directory,
    environment_report,
    load_notebook_config,
    load_stage_registry,
    registry_dataframe,
    run_command,
    run_stage,
    stage_by_id,
    validate_stage_registry,
    write_run_summary,
)

print(f"Repository: {REPO_ROOT}")

## 1. Notebook parameters

These values are loaded from `configs/master_notebook.yaml`. The cell is tagged `parameters` so it can later be automated with tools such as Papermill, but no such automation is required now.

Recommended modes:

- `status`: inspect only; safest and current default.
- `selected`: execute only the stage IDs listed in `SELECTED_STAGES`.
- `sentinel`: reserved for a small validated pilot.
- `full`: reserved for a frozen pipeline and still subject to `full_run_enabled`.

In [ ]:
NOTEBOOK_CONFIG = load_notebook_config(REPO_ROOT)

EXECUTION_MODE = NOTEBOOK_CONFIG.get("execution_mode", "status")
SELECTED_STAGES = [str(x).zfill(2) for x in NOTEBOOK_CONFIG.get("selected_stages", [])]
RUN_PREFLIGHT_TESTS = bool(NOTEBOOK_CONFIG.get("run_preflight_tests", False))
STOP_ON_FAILURE = bool(NOTEBOOK_CONFIG.get("stop_on_failure", True))
ALLOW_STUB_EXECUTION = bool(NOTEBOOK_CONFIG.get("allow_stub_execution", False))
WRITE_RUN_SUMMARY = bool(NOTEBOOK_CONFIG.get("write_run_summary", True))

VALID_MODES = {"status", "selected", "sentinel", "full"}
assert EXECUTION_MODE in VALID_MODES, f"Invalid execution mode: {EXECUTION_MODE}"

EXECUTE_PIPELINE = EXECUTION_MODE in {"selected", "sentinel", "full"}
print({
    "execution_mode": EXECUTION_MODE,
    "selected_stages": SELECTED_STAGES,
    "run_preflight_tests": RUN_PREFLIGHT_TESTS,
    "execute_pipeline": EXECUTE_PIPELINE,
    "allow_stub_execution": ALLOW_STUB_EXECUTION,
})

## 2. Project identity and safety state

This cell records the current Python environment, repository commit, schema version, observation window, and full-run gate. These values belong in every meaningful run record.

In [ ]:
ENVIRONMENT = environment_report(REPO_ROOT)
display(pd.DataFrame([ENVIRONMENT]).T.rename(columns={0: "value"}))

## 3. Load and validate the canonical pipeline registry

`configs/pipeline_stages.yaml` is the single source of truth for stage order and readiness. Every stage must identify:

- a stable ID and phase;
- the numbered script that runs it;
- the governing spec;
- expected inputs and outputs;
- its acceptance gate;
- its current status (`stub`, `sentinel`, `ready`, or `frozen`).

In [ ]:
STAGE_REGISTRY = load_stage_registry(REPO_ROOT)
REGISTRY_FINDINGS = validate_stage_registry(REPO_ROOT, STAGE_REGISTRY)

if REGISTRY_FINDINGS:
    display(pd.DataFrame(REGISTRY_FINDINGS))
    raise RuntimeError("Pipeline registry validation failed.")

STAGE_TABLE = registry_dataframe(REPO_ROOT, STAGE_REGISTRY)
display(STAGE_TABLE)

## 4. Preflight checks

Before any real stage execution, the project should verify:

- repository contamination controls;
- JSON-schema validity;
- temporal-policy tests;
- stage-registry integrity;
- package imports;
- clean and explicit configuration.

`RUN_PREFLIGHT_TESTS` is `false` by default so opening the notebook remains quick. Turn it on when preparing a sentinel or production run.

In [ ]:
PREFLIGHT_COMMANDS = {
    "all_tests": [sys.executable, "-m", "pytest", "-q"],
    "contamination": [sys.executable, "-m", "pytest", "-q", "tests/contamination"],
    "schema": [sys.executable, "-m", "dynamic_ai_products.validation", "schemas"],
    "workflow_registry": [sys.executable, "-m", "dynamic_ai_products.workflow"],
}

preflight_rows = []
for name, command in PREFLIGHT_COMMANDS.items():
    result = run_command(command, repo_root=REPO_ROOT, execute=RUN_PREFLIGHT_TESTS)
    preflight_rows.append({
        "check": name,
        "command": " ".join(command),
        "executed": result.executed,
        "returncode": result.returncode,
        "blocked_reason": result.blocked_reason,
    })

PREFLIGHT_RESULTS = pd.DataFrame(preflight_rows)
display(PREFLIGHT_RESULTS)

## 5. Create an immutable notebook run directory when executing

A real execution gets a unique folder under `data/runs/`. Dry status runs do not create files. The directory stores command logs and the final structured summary; it must never be reused or overwritten.

In [ ]:
RUN_DIR = None
if EXECUTE_PIPELINE:
    RUN_DIR = create_notebook_run_directory(REPO_ROOT)
    print(f"Run directory: {RUN_DIR}")
else:
    print("Status mode: no run directory created.")

# Phase A — Company universe and official-source corpus

This phase establishes what firms and dates are studied, discovers eligible official sources, snapshots them immutably, and converts them into normalized evidence-addressable passages.

It must be completed before product extraction. A model cannot produce a defensible longitudinal universe from an uncontrolled or temporally mixed document collection.

## Stage 00 — Build the company universe

**Script:** `pipelines/00_build_company_universe.py`  
**Spec:** `specs/SPEC-001-company-universe.md`

The output should contain stable firm IDs, CIK/ticker metadata, inclusion status, and the observation window. Universe decisions must be explicit rather than inferred during downstream extraction.

## Stages 01–02 — Discover SEC and official-web sources

SEC and official company materials serve different functions:

- SEC filings provide standardized annual and quarterly disclosure.
- Investor-relations materials provide launch and commercialization signals.
- Product pages describe customer-facing functions.
- Developer documentation and release notes reveal action/execution depth and availability.
- Historical snapshots prevent current webpages from leaking into earlier observations.

Discovery creates candidates; it does not yet treat every candidate as valid evidence.

## Stages 03–04 — Snapshot and normalize

Raw source files are immutable. Normalized text and passages are derived artifacts with stable links back to source hashes, dates, URLs/accessions, and offsets. A quoted evidence string must be recoverable from a specific passage.

In [ ]:
PHASE_A = ["00", "01", "02", "03", "04"]
display(STAGE_TABLE[STAGE_TABLE["stage"].isin(PHASE_A)])

# Phase B — Product, capability, and customer-task extraction

The hierarchy is:

```text
Company → Product family → Product → Capability → Customer-facing task
```

- A **product** is an identifiable commercial offering.
- A **capability** is a concrete function the product provides.
- A **task** is the customer's economically meaningful intended work outcome.

Extraction and measurement remain separate. This phase must not decide whether a task is replicable, defensible, or strategically advantageous.

## Stage 05 — Product extraction

Use a high-recall discovery pass followed by high-precision consolidation. Product names, bundles, product families, features, and generic platforms must not be conflated.

Typical failure cases become eval cases:

- suite mistaken for a separately sold product;
- discontinued product treated as active;
- future roadmap treated as available;
- one product duplicated under naming variants.

## Stage 06 — Capability extraction

A phrase such as “AI-powered platform” is not a capability. The source must describe what the product concretely enables the customer to do.

Examples:

- `AI-powered document intelligence` → not sufficient alone.
- `Ask questions across documents and receive cited answers` → concrete capability/task evidence.
- `Agent creates incidents, updates records, and escalates exceptions` → concrete workflow action evidence.

## Stage 07 — Task extraction and consolidation

Tasks are written from the customer perspective, generally as **verb + object + intended outcome**. The pipeline should preserve meaningful differences without producing one task for every sentence.

Discovery prioritizes recall. Consolidation handles:

- duplicates;
- overly broad tasks;
- microscopic steps;
- capability/task confusion;
- unsupported inferences;
- non-customer-facing internal work.

## Stage 08 — Task role classification

Role is classified after task extraction:

- `core`: central to the product's commercial value proposition;
- `major_supporting`: materially supports the product's value;
- `peripheral`: convenience or secondary functionality.

Role is an evidence-backed judgment and later informs aggregation. It must not be guessed merely because a task appears early in a webpage.

In [ ]:
PHASE_B = ["05", "06", "07", "08"]
display(STAGE_TABLE[STAGE_TABLE["stage"].isin(PHASE_B)])

# Phase C — Longitudinal identity and task transitions

Yearly extraction creates observations, not yet a longitudinal history. This phase determines whether observations represent the same economic task, a transformed continuation, a split/merge, a replacement, a new task, or a discontinued task.

## Stage 09 — Match entities over time

Candidate generation can use names, product relationships, capability overlap, and customer-need similarity. Final adjudication asks whether two records serve the same underlying customer need.

The system preserves both:

- stable longitudinal identity; and
- the exact yearly wording and evidence.

## Stage 10 — Classify task transitions

Supported labels include:

`same_task`, `renamed`, `expanded`, `contracted`, `ai_assisted`, `workflow_integrated`, `agentified`, `split`, `merged`, `replaced`, `discontinued`, `new_task`, and `uncertain`.

Task inflation is a major risk: an AI feature should not automatically become a “new task” when it simply changes how the same customer need is delivered.

In [ ]:
PHASE_C = ["09", "10"]
display(STAGE_TABLE[STAGE_TABLE["stage"].isin(PHASE_C)])

# Phase D — Dated frontier baseline and task-level measurement

Measurement is performed only after the task universe and longitudinal links are validated. Each observation uses the frontier capabilities actually available at its cutoff date.

The project keeps multiple constructs separate rather than forcing them immediately into one score.

## Stage 11 — Build the frontier registry

For each dated frontier release, record availability, modalities, context, tool/API access, relevant evidence, and capability limitations. Release names alone are not sufficient.

A later model must never be used to score an earlier filing date. The registry exists to prevent that leakage.

## Stage 12 — Measure task-years

Four main measurement families are currently planned:

1. **Frontier task replicability** — can the dated frontier system satisfy the customer's core task without the focal firm's product?
2. **AI transformation depth** — did AI provide peripheral assistance, direct output generation, native workflow integration, multi-step execution, or agentic orchestration?
3. **Deployment scale** — was the capability roadmap-only, limited, generally available, cross-product, customer-deployed, or commercially material?
4. **Task-specific defensibility** — does the product retain meaningful data, workflow state, execution authority, specialized-engine, governance, integration, or switching-friction advantages?

Deep adoption is not automatically advantageous. A high-replicability task can remain commoditized even after serious AI integration, as the Chegg-style counterexample illustrates.

In [ ]:
PHASE_D = ["11", "12"]
display(STAGE_TABLE[STAGE_TABLE["stage"].isin(PHASE_D)])

# Phase E — Aggregation and analysis

Task-level data is the empirical foundation. Aggregation is a declared transformation, not a replacement for the underlying observations.

## Stage 13 — Aggregate product- and firm-year metrics

Aggregation may use core/supporting task roles and product structure. Every reported aggregate should be decomposable into task contributions and accompanied by sensitivity variants.

Entry, exit, and transformation should remain distinguishable:

- existing tasks became more replicable;
- firms introduced new AI-native tasks;
- tasks were retired or replaced;
- the product mix shifted.

## Stage 14 — Analysis outputs

Initial outputs should prioritize:

- source-coverage and extraction-quality diagnostics;
- product/task trajectories;
- transition matrices;
- firm archetypes in replicability × transformation × defensibility space;
- evidence-grounded case studies;
- robustness and source-ablation comparisons.

Panel or outcome analysis can follow, but post-shock adoption is endogenous and should not be described as causal without a credible identification strategy.

In [ ]:
PHASE_E = ["13", "14"]
display(STAGE_TABLE[STAGE_TABLE["stage"].isin(PHASE_E)])

# Eval gates and prompt-change control

A stage is not improved by repeatedly editing a prompt until a few examples look good. The required development loop is:

```text
Observed failure
→ create permanent eval case
→ document expected behavior
→ open change request
→ modify one general rule
→ run dev + regression + frozen tests
→ compare fixed cases and new regressions
→ accept or reject
```

Hard gates include schema validity, valid source IDs, quote support, no future-date evidence, no legacy contamination, and no silent unknown-to-guessed conversion.

In [ ]:
EVAL_DOCUMENTS = [
    "evals/EVAL_HARNESS.md",
    "evals/CHANGE_CONTROL_PROTOCOL.md",
    "docs/methodology/PROMPT_DEVELOPMENT_AND_EVALUATION_PROTOCOL.md",
    "specs/SPEC-020-evaluation-harness.md",
    "specs/SPEC-022-evaluation-data-model.md",
    "specs/SPEC-023-deterministic-validation.md",
    "specs/SPEC-024-run-versioning-and-comparison.md",
]

pd.DataFrame({
    "document": EVAL_DOCUMENTS,
    "exists": [(REPO_ROOT / path).exists() for path in EVAL_DOCUMENTS],
})

# Stage execution controller

The next cell is the only generic stage launcher in the notebook. It builds each command from the canonical registry and records structured outcomes.

Safety behavior:

- `status` mode: nothing executes.
- `selected` mode: only `SELECTED_STAGES` execute.
- stub stages are blocked unless `ALLOW_STUB_EXECUTION=True`.
- aggregation and analysis remain blocked while `full_run_enabled=false`.
- a non-zero return code can stop subsequent stages.

In [ ]:
def stages_for_current_mode() -> list[str]:
    all_ids = [row["id"] for row in STAGE_REGISTRY["stages"]]
    if EXECUTION_MODE == "status":
        return []
    if EXECUTION_MODE == "selected":
        unknown = sorted(set(SELECTED_STAGES) - set(all_ids))
        if unknown:
            raise ValueError(f"Unknown selected stages: {unknown}")
        return SELECTED_STAGES
    if EXECUTION_MODE == "sentinel":
        return [row["id"] for row in STAGE_REGISTRY["stages"] if row["status"] in {"sentinel", "ready", "frozen"}]
    if EXECUTION_MODE == "full":
        return all_ids
    raise ValueError(EXECUTION_MODE)

STAGES_TO_RUN = stages_for_current_mode()
print(f"Stages selected for execution: {STAGES_TO_RUN or 'none'}")

In [ ]:
STAGE_RESULTS = []

for stage_id in STAGES_TO_RUN:
    stage = stage_by_id(STAGE_REGISTRY, stage_id)
    print(f"[{stage_id}] {stage['name']} — status={stage['status']}")
    result = run_stage(
        stage_id,
        repo_root=REPO_ROOT,
        registry=STAGE_REGISTRY,
        execute=True,
        allow_stub_execution=ALLOW_STUB_EXECUTION,
        log_dir=RUN_DIR,
    )
    STAGE_RESULTS.append({
        "stage": stage_id,
        "name": stage["name"],
        "status": stage["status"],
        "executed": result.executed,
        "returncode": result.returncode,
        "blocked_reason": result.blocked_reason,
        "stdout_tail": result.stdout[-500:],
        "stderr_tail": result.stderr[-500:],
    })
    if result.blocked_reason:
        print(f"  BLOCKED: {result.blocked_reason}")
    elif result.returncode == 0:
        print("  PASS")
    else:
        print(f"  FAIL returncode={result.returncode}")
        if STOP_ON_FAILURE:
            break

STAGE_RESULTS_DF = pd.DataFrame(STAGE_RESULTS)
display(STAGE_RESULTS_DF if not STAGE_RESULTS_DF.empty else pd.DataFrame({"status": ["No stages executed in status mode."]}))

# Expected outputs and completion audit

This section does not assume that a listed path already exists. It provides a checklist derived from the registry so the researcher can distinguish:

- stage not implemented;
- stage implemented but not run;
- stage run but expected output absent;
- stage output present and ready for validation.

In [ ]:
output_rows = []
for stage in STAGE_REGISTRY["stages"]:
    for output in stage["outputs"]:
        # Wildcard declarations are documentation patterns, not single paths.
        is_pattern = "*" in output
        exists = None if is_pattern else (REPO_ROOT / output).exists()
        output_rows.append({
            "stage": stage["id"],
            "name": stage["name"],
            "declared_output": output,
            "is_pattern": is_pattern,
            "exists": exists,
        })
OUTPUT_AUDIT = pd.DataFrame(output_rows)
display(OUTPUT_AUDIT)

# Write the run summary

When execution is enabled, the final summary captures the environment, configuration, registry version, selected stages, preflight results, and stage outcomes. This summary is an audit artifact, not a substitute for each stage's own detailed manifest.

In [ ]:
RUN_SUMMARY_PATH = None
if RUN_DIR is not None and WRITE_RUN_SUMMARY:
    summary = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "environment": ENVIRONMENT,
        "notebook_config": NOTEBOOK_CONFIG,
        "registry_version": STAGE_REGISTRY.get("registry_version"),
        "selected_stages": STAGES_TO_RUN,
        "preflight": PREFLIGHT_RESULTS.to_dict(orient="records"),
        "stage_results": STAGE_RESULTS,
    }
    RUN_SUMMARY_PATH = write_run_summary(RUN_DIR, summary)
    print(f"Wrote: {RUN_SUMMARY_PATH}")
else:
    print("No run summary written in status mode.")

# Current repository state and next implementation sequence

Stage readiness is mixed, and this notebook should not flatten it in either
direction. Two facts hold at the same time:

- **Stage 00 corpus construction is complete and governed.** The acquisition
  lineage, issuer exclusions and Item 1 packet build have all run under named,
  hash-bound manifests, and the canonical v5 packet corpus of 7,042 packets plus
  530 recorded packet failures exists on disk. `pipelines/00_build_company_universe.py`
  is a real, implemented entry point with many governed modes, not a placeholder.
- **The high-recall screen has no authoritative release.** The instrument is
  built through ADR-118, but the most recent full-cohort run stopped partway and
  left a failure receipt, so no `SCREEN_v1` exists. Its governed continuation is
  pending and requires fresh governance plus a separate authorization.

Later numbered entry points remain unimplemented and raise `NotImplementedError`
where that is still true. That is an explicit protection against pretending a
designed stage is production-ready, and it is a per-stage fact rather than a
statement about the repository as a whole. The classifier, deterministic tiers,
`UNIVERSE_v1`, PCT and the econometric stages are **not** implemented or runnable.

For per-stage readiness, read `configs/pipeline_stages.yaml` and the registry
table printed above rather than this prose. The remaining implementation order:

1. governed high-recall continuation, release audit, and `SCREEN_v1`;
2. classifier successor, deterministic tiers, and `UNIVERSE_v1`;
3. product/capability/task extraction sentinel (05–08);
4. longitudinal matching (09–10);
5. frontier registry and measurements (11–12);
6. aggregation and analysis (13–14).

After each stage is implemented, add tests and eval cases, run the relevant
suite, then update its registry status conservatively.

## Session-closing checklist

Before closing a work session:

- [ ] Record methodological decisions in `docs/DECISION_LOG.md`.
- [ ] Convert every newly observed failure into an eval or regression case.
- [ ] Do not edit original run outputs.
- [ ] Confirm no future-date evidence entered historical observations.
- [ ] Run relevant tests.
- [ ] Inspect `git diff`.
- [ ] Commit code, prompt, schema, spec, and eval changes together when they belong to the same decision.
- [ ] Update stage readiness only when acceptance criteria are met.